# Perfume Authenticity Classifier — v3 (MobileNetV3-Large + Advanced Anti-Overfit)

**Model change:** EfficientNet-B0 → **MobileNetV3-Large**  
MobileNetV3 uses Squeeze-and-Excitation (SE) blocks and a Hard-Swish activation that generalise better on small packaging datasets, has fewer parameters (5.5 M vs 5.3 M but a lighter feature extractor), and trains significantly faster — important when you have only ~170–220 images.

**Anti-overfit techniques used:**
- Heavy augmentation (RandAugment + geometric + colour jitter)
- MixUp training
- Stochastic Depth (Drop-path) — already baked into MobileNetV3 head
- Label smoothing loss
- Aggressive dropout (0.5 in head)
- Cosine-annealing LR with warm restarts
- Two-phase training (frozen backbone → full fine-tune)
- Grad-CAM visualisation to verify the model looks at packaging, not background

**Dataset:** 4 zip files — `original.zip`, `fake.zip`, `N-original.zip`, `N-fake.zip`  
Supports HEIC images automatically via `pillow-heif`.

## 0. Setup & Install Dependencies

In [1]:
# Install required libraries (run once)
!pip install pillow-heif grad-cam timm -q

In [2]:
import os, zipfile, shutil, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
import pillow_heif

# Register HEIC/HEIF support so PIL can open .HEIC files
pillow_heif.register_heif_opener()

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


## 1. Mount Drive & Extract Dataset

Expected layout in your Google Drive:
```
MyDrive/
  perfume_data/
    original.zip
    fake.zip
    N-original.zip
    N-fake.zip
```

In [7]:
import os
import shutil

# ── Configuration ─────────────────────────────────────────────
BASE_DIR = "Dataset Commercial fraud"

SOURCE_FOLDERS = {
    "original": "new_original",
    "N-original": "new_original",
    "fake": "new_fake",
    "N-fake": "new_fake",
}

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.heic', '.heif', '.webp'}

# ── Create output folders ─────────────────────────────────────
for target in set(SOURCE_FOLDERS.values()):
    os.makedirs(os.path.join(BASE_DIR, target), exist_ok=True)

# ── Merge datasets ────────────────────────────────────────────
for src_folder, target_folder in SOURCE_FOLDERS.items():
    src_path = os.path.join(BASE_DIR, src_folder)
    dst_path = os.path.join(BASE_DIR, target_folder)

    copied = 0

    for root, _, files in os.walk(src_path):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()

            if ext not in VALID_EXTENSIONS:
                continue

            src_file = os.path.join(root, fname)

            # Avoid overwrite → make unique name
            unique_name = f"{src_folder}_{fname}"
            dst_file = os.path.join(dst_path, unique_name)

            shutil.copy2(src_file, dst_file)
            copied += 1

    print(f"{src_folder} → {target_folder} | Copied: {copied}")

# ── Final count ───────────────────────────────────────────────
for target in ["new_original", "new_fake"]:
    path = os.path.join(BASE_DIR, target)
    total = len(os.listdir(path))
    print(f"Total {target}: {total} images")

original → new_original | Copied: 0
N-original → new_original | Copied: 0
fake → new_fake | Copied: 0
N-fake → new_fake | Copied: 0
Total new_original: 0 images
Total new_fake: 0 images


## 2. Custom Dataset with HEIC Support

In [6]:
class PerfumeDataset(Dataset):
    """
    Custom dataset that:
    - Loads from two folders: DATA_DIR/original and DATA_DIR/fake
    - Handles HEIC/HEIF via pillow-heif
    - Skips corrupt files gracefully
    """
    def __init__(self, file_list, labels, transform=None):
        self.file_list = file_list
        self.labels    = labels
        self.transform = transform
        self.classes   = ['fake', 'original']   # alphabetical — same as ImageFolder
        self.class_to_idx = {'fake': 0, 'original': 1}

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path  = self.file_list[idx]
        label = self.labels[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            # Return a blank image if file is corrupt
            img = Image.new('RGB', (224, 224), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, label


def build_file_list(data_dir):
    """Scan data_dir/fake and data_dir/original, return lists of paths and int labels."""
    paths, labels = [], []
    class_to_idx = {'fake': 0, 'original': 1}
    for cls_name, cls_idx in class_to_idx.items():
        cls_dir = os.path.join(data_dir, cls_name)
        for fname in sorted(os.listdir(cls_dir)):
            ext = os.path.splitext(fname)[1].lower()
            if ext not in VALID_EXTENSIONS:
                continue
            paths.append(os.path.join(cls_dir, fname))
            labels.append(cls_idx)
    return paths, labels


all_paths, all_labels = build_file_list(DATA_DIR)
print(f'Total images found: {len(all_paths)}')
print('Class distribution:', Counter(all_labels))

Total images found: 0
Class distribution: Counter()


## 3. Transforms
**RandAugment** applies random combinations of photometric and geometric augmentations — proven to dramatically improve generalisation on small datasets.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE      = 224

train_transform = transforms.Compose([
    # Step 1: Resize to slightly larger than crop target
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(IMG_SIZE),

    # Step 2: Geometric augmentations
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),

    # Step 3: Colour augmentations — packaging appearance varies under lighting
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.08),
    transforms.RandomGrayscale(p=0.05),

    # Step 4: RandAugment — N=2 operations, M=9 magnitude
    transforms.RandAugment(num_ops=2, magnitude=9),

    # Step 5: Random erasing (simulates occlusion/reflections on packaging)
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15), ratio=(0.3, 3.0)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Transforms defined.')

## 4. Stratified Split + Weighted Sampler

In [ ]:
all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

# Stratified 70 / 15 / 15
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=42
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=42
)

print(f'Train: {len(train_paths)}  |  Val: {len(val_paths)}  |  Test: {len(test_paths)}')
print('Train class dist:', Counter(train_labels.tolist()))
print('Val   class dist:', Counter(val_labels.tolist()))
print('Test  class dist:', Counter(test_labels.tolist()))

# Dataset objects
train_ds = PerfumeDataset(train_paths.tolist(), train_labels.tolist(), transform=train_transform)
val_ds   = PerfumeDataset(val_paths.tolist(),   val_labels.tolist(),   transform=val_transform)
test_ds  = PerfumeDataset(test_paths.tolist(),  test_labels.tolist(),  transform=val_transform)

# Weighted sampler to handle class imbalance
class_counts  = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(train_ds),
    replacement=True
)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

## 5. Model — MobileNetV3-Large

**Why MobileNetV3-Large instead of EfficientNet-B0?**
| Property | EfficientNet-B0 | MobileNetV3-Large |
|---|---|---|
| Params | 5.3 M | 5.5 M |
| Top-1 ImageNet | 77.7% | 75.3% |
| Inference speed | Medium | **Very fast** |
| SE blocks | ✓ | ✓ |
| Hard-Swish | ✗ | ✓ (better gradients) |
| Small-dataset generalisation | Good | **Better** (empirically) |

MobileNetV3's Hard-Swish activation and SE blocks give it strong texture discrimination — critical for reading printing quality differences between genuine and counterfeit packaging.

In [ ]:
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

def build_model(num_classes=2, dropout=0.5):
    model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)

    # Freeze backbone for Phase 1
    for param in model.parameters():
        param.requires_grad = False

    # Replace classifier head
    # MobileNetV3-Large classifier: [Linear(960,1280), Hardswish, Dropout, Linear(1280,1000)]
    in_features = model.classifier[0].in_features    # 960
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.Hardswish(),
        nn.Dropout(p=dropout),
        nn.Linear(512, 256),
        nn.Hardswish(),
        nn.Dropout(p=dropout * 0.6),
        nn.Linear(256, num_classes)
    )
    return model


model = build_model(num_classes=2, dropout=0.5).to(device)

total_params   = sum(p.numel() for p in model.parameters())
trainable_p1   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:       {total_params:,}')
print(f'Trainable (Phase1): {trainable_p1:,}')

## 6. MixUp Helper
MixUp linearly interpolates pairs of images and their labels. This prevents the model from memorising training examples and forces smooth decision boundaries — a proven anti-overfit technique.

In [ ]:
def mixup_data(x, y, alpha=0.4):
    """Returns mixed inputs, pairs of targets, and lambda."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('MixUp helpers ready.')

## 7. Training Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, use_mixup=True, mixup_alpha=0.4):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        if use_mixup and np.random.random() > 0.5:
            inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=mixup_alpha)
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        # Gradient clipping: prevents exploding gradients during full fine-tune
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)
        total_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def run_training(model, train_loader, val_loader, epochs, lr, label='',
                 use_mixup=True, patience=10, save_path='/content/best_model.pt'):
    """
    Label smoothing loss: reduces overconfidence, improves calibration.
    CosineAnnealingWarmRestarts: escapes local minima on small datasets.
    """
    # Label smoothing dampens over-confident predictions → reduces overfitting
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=5e-4
    )

    # Warm restarts: T_0=10 → restart every 10 epochs, T_mult=2 doubles each cycle
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler    = torch.cuda.amp.GradScaler()

    best_val_acc, patience_counter = 0.0, 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer,
                                          scaler, use_mixup=use_mixup)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        gap = tr_acc - vl_acc   # overfit indicator
        gap_flag = ' ⚠ overfit?' if gap > 0.12 else ''
        print(f"[{label}] Ep {epoch:02d}/{epochs} | "
              f"Train {tr_acc:.3f} ({tr_loss:.4f}) | "
              f"Val {vl_acc:.3f} ({vl_loss:.4f}) | "
              f"Gap {gap:+.3f}{gap_flag}")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
            print(f'  ✓ New best val acc: {best_val_acc:.3f} — saved checkpoint')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping triggered at epoch {epoch}')
                break

    print(f'\nBest val acc [{label}]: {best_val_acc:.3f}')
    return history

## 8. Phase 1 — Train Head Only
Freeze the backbone for the first 15 epochs to warm up the new head without corrupting pretrained features.

In [ ]:
print('Trainable params (Phase 1):', sum(p.numel() for p in model.parameters() if p.requires_grad))

history1 = run_training(
    model, train_loader, val_loader,
    epochs=15,
    lr=1e-3,
    label='Phase1-HeadOnly',
    use_mixup=False,    # MixUp less effective when only head trains
    patience=8
)

## 9. Phase 2 — Full Fine-Tuning
Unfreeze the entire backbone. Use a 10× lower learning rate + differential learning rates: smaller LR for lower layers (they need less adjustment), higher LR for the head.

In [ ]:
# Unfreeze everything — but use differential learning rates
for param in model.parameters():
    param.requires_grad = True

print('Trainable params (Phase 2):', sum(p.numel() for p in model.parameters() if p.requires_grad))

# Load best head-only checkpoint before fine-tuning
model.load_state_dict(torch.load('/content/best_model.pt'))
model = model.to(device)

history2 = run_training(
    model, train_loader, val_loader,
    epochs=30,
    lr=5e-5,           # 20× lower than Phase 1
    label='Phase2-FineTune',
    use_mixup=True,    # MixUp ON for full fine-tune
    patience=12,
    save_path='/content/best_model.pt'
)

## 10. Test Set Evaluation

In [ ]:
# Load the best checkpoint
model.load_state_dict(torch.load('/content/best_model.pt'))
model.eval()

all_preds, all_labels_test, all_probs = [], [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels_test.extend(labels.numpy())
        all_probs.extend(probs)

class_names = ['fake', 'original']
print('=' * 60)
print('CLASSIFICATION REPORT — TEST SET')
print('=' * 60)
print(classification_report(all_labels_test, all_preds, target_names=class_names))
print(f'AUC-ROC: {roc_auc_score(all_labels_test, all_probs):.4f}')

# Confusion matrix
cm = confusion_matrix(all_labels_test, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix — Test Set', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Training Curves (Overfitting Check)
A healthy model has train/val curves that track closely. A widening gap signals overfitting.

In [ ]:
all_hist = {k: history1[k] + history2[k] for k in history1}
phase1_len = len(history1['train_loss'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(all_hist['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(all_hist['val_loss'],   label='Val Loss',   linewidth=2)
ax1.axvline(phase1_len, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
ax1.fill_between(range(len(all_hist['train_loss'])),
                 all_hist['train_loss'], all_hist['val_loss'], alpha=0.08)
ax1.set_title('Loss per Epoch', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(all_hist['train_acc'], label='Train Acc', linewidth=2)
ax2.plot(all_hist['val_acc'],   label='Val Acc',   linewidth=2)
ax2.axvline(phase1_len, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
ax2.fill_between(range(len(all_hist['train_acc'])),
                 all_hist['train_acc'], all_hist['val_acc'], alpha=0.08)
ax2.set_title('Accuracy per Epoch', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('MobileNetV3-Large — Perfume Authenticity Classifier', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to /content/training_curves.png')

## 12. Grad-CAM — Visual Explainability
Verifies that the model focuses on the packaging label/cap region rather than background.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# MobileNetV3-Large target layer: last conv block before adaptive pool
target_layers = [model.features[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

n_vis = min(6, len(test_ds))
fig, axes = plt.subplots(2, n_vis, figsize=(3 * n_vis, 6))
fig.suptitle('Grad-CAM — What the model looks at', fontsize=14)

model.eval()
mean_t = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std_t  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

for i in range(n_vis):
    img_tensor, label = test_ds[i]
    input_t = img_tensor.unsqueeze(0).to(device)
    pred    = model(input_t).argmax(1).item()

    grayscale_cam = cam(input_tensor=input_t,
                        targets=[ClassifierOutputTarget(pred)])[0]

    # Denormalise for display
    img_show = (img_tensor * std_t + mean_t).permute(1, 2, 0).numpy().clip(0, 1)
    cam_img  = show_cam_on_image(img_show, grayscale_cam, use_rgb=True)

    true_name = class_names[label]
    pred_name = class_names[pred]
    colour    = 'green' if pred == label else 'red'

    axes[0, i].imshow(img_show)
    axes[0, i].set_title(f'GT: {true_name}', fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(cam_img)
    axes[1, i].set_title(f'Pred: {pred_name}', fontsize=9, color=colour)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('/content/gradcam.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to /content/gradcam.png')

## 13. Save Model for Deployment

In [ ]:
# Save full model info for deployment
torch.save({
    'model_state_dict' : model.state_dict(),
    'class_to_idx'     : {'fake': 0, 'original': 1},
    'architecture'     : 'mobilenet_v3_large',
    'img_size'         : IMG_SIZE,
    'imagenet_mean'    : IMAGENET_MEAN,
    'imagenet_std'     : IMAGENET_STD,
}, '/content/perfume_classifier_v3.pt')

# Copy to Drive for persistence
shutil.copy('/content/perfume_classifier_v3.pt',
            os.path.join(DRIVE_FOLDER, 'perfume_classifier_v3.pt'))
print('Model saved to Drive.')

## 14. Single-Image Inference Function
Ready to use on a new smartphone photo.

In [ ]:
def predict_image(image_path, model, threshold=0.5):
    """
    Predict whether a perfume image is original or fake.
    Returns: dict with prediction, confidence, and class probabilities.
    """
    model.eval()
    img = Image.open(image_path).convert('RGB')   # HEIC handled by pillow-heif
    tensor = val_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=1).cpu().numpy()[0]

    idx   = int(probs.argmax())
    label = class_names[idx]
    conf  = float(probs[idx])

    result = {
        'prediction'   : label,
        'confidence'   : conf,
        'prob_fake'    : float(probs[0]),
        'prob_original': float(probs[1]),
    }
    print(f"Prediction : {label.upper()}")
    print(f"Confidence : {conf:.1%}")
    print(f"Fake prob  : {probs[0]:.1%}  |  Original prob: {probs[1]:.1%}")
    return result


# ── Quick demo on first test image ──
sample_path = test_ds.file_list[0]
print(f'Testing on: {os.path.basename(sample_path)}')
result = predict_image(sample_path, model)